# Hyperparameter Tuning for Inner-Loop Gaussian Priors

Grid-search `(sigma, tau, nu)` on the seed structure `routine + affinity`.
Fits on the training split, scores on the validation split using *unweighted*
held-out log-likelihood (no parameter prior, no structure prior).

Inputs (produced by `01_data_preparation.ipynb` and `02_dsl_features.ipynb`):
- `data/purchases_processed.parquet`
- `data/train_choices.pkl`
- `data/val_choices.pkl`
- `data/train_features.pkl` (gives train features + event_weights if present)

Outputs:
- `data/tuned_hyperparams.json`
- `data/tuning_sensitivity.png`

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.dsl import DSLStructure
from src.tuning import tune_hyperparameters, val_log_likelihood
from src.inner_loop import fit_weights

DATA_DIR = Path('../data')

## 1. Load train features + choices, rebuild val features

`02_dsl_features.ipynb` only persists train features. We rebuild validation features
here using the same lookup-table pattern so the val split uses the same
feature definitions as train.

In [ ]:
with open(DATA_DIR / 'train_features.pkl', 'rb') as f:
    train_feat = pickle.load(f)

train_features_list  = train_feat['features_list']
train_chosen_indices = train_feat['chosen_indices']
train_customer_ids   = train_feat['customer_ids']
train_categories     = train_feat['categories']
structure_saved      = DSLStructure.from_dict(train_feat['structure'])

# Event weights (from the leverage-score subsample in 01). Optional.
train_event_weights = None
ewm_path = DATA_DIR / 'event_weight_map.pkl'
if ewm_path.exists():
    with open(ewm_path, 'rb') as f:
        ew_map = pickle.load(f)
    train_event_weights = np.array([ew_map.get(cid, 1.0) for cid in train_customer_ids])
    print(f'Loaded event_weight_map: {len(ew_map)} customers, '
          f'train weight sum={train_event_weights.sum():.0f}')
else:
    print('No event_weight_map.pkl — using uniform train weights.')

print(f'Train events: {len(train_features_list):,}')
print(f'Saved structure: {structure_saved}  (tuning will use the seed structure below)')

In [ ]:
seed_structure = DSLStructure.initial()
print(f'Seed structure: {seed_structure}')
print(f'  terms: {[t.display_name for t in seed_structure.terms]}')

# The feature columns saved in train_features.pkl correspond to `structure_saved`.
# We need to slice out the columns corresponding to the seed structure's terms.
saved_names = [t.display_name for t in structure_saved.terms]
seed_names  = [t.display_name for t in seed_structure.terms]
missing = [n for n in seed_names if n not in saved_names]
if missing:
    raise RuntimeError(
        f'Seed structure terms {missing} are not present in the saved '
        f'train_features.pkl columns {saved_names}. Re-run 02_dsl_features.ipynb '
        f'with at least the seed structure terms included.'
    )
col_idx = [saved_names.index(n) for n in seed_names]
train_features_list = [f[:, col_idx] for f in train_features_list]
print(f'Train features sliced to seed columns: shape per event = {train_features_list[0].shape}')

## 2. Build validation features for the seed structure

Mirrors the lookup-table pattern in `02_dsl_features.ipynb`. The seed structure
only needs `routine` and `affinity`, so we compute only those columns for val events.

In [ ]:
df = pd.read_parquet(DATA_DIR / 'purchases_processed.parquet')
with open(DATA_DIR / 'val_choices.pkl', 'rb') as f:
    val_choices = pickle.load(f)

# Routine per (customer, asin): last known purchase_count state at purchase time.
cust_item_df = (
    df.sort_values(['customer_id', 'asin', 'order_date'])
      .groupby(['customer_id', 'asin'])
      .last()[['routine']]
)
cust_item_routine = cust_item_df['routine'].to_dict()

# Category affinity per (customer, category): last known cat_affinity count.
cust_cat_affinity = (
    df.sort_values(['customer_id', 'category', 'order_date'])
      .groupby(['customer_id', 'category'])['cat_affinity']
      .last()
      .to_dict()
)

# ASIN → category (for mapping alternatives to a category)
asin_category = df.groupby('asin')['category'].first().to_dict()

print(f'Val events: {len(val_choices):,}')
print(f'cust_item_routine: {len(cust_item_routine):,}  cust_cat_affinity: {len(cust_cat_affinity):,}')

In [ ]:
def build_val_feature_matrix(event):
    """Build [n_alts, 2] feature matrix for seed structure (routine, affinity)."""
    cid = event['customer_id']
    event_cat = event['category']
    asins = event['choice_asins']
    n_alt = len(asins)
    mat = np.zeros((n_alt, 2))
    for k, asin in enumerate(asins):
        routine_val = float(cust_item_routine.get((cid, asin), 0.0))
        cat = asin_category.get(asin, event_cat)
        affinity_val = np.log1p(cust_cat_affinity.get((cid, cat), 0.0))
        mat[k, 0] = routine_val
        mat[k, 1] = affinity_val
    return mat

val_features_list  = []
val_chosen_indices = []
val_customer_ids   = []
val_categories     = []
for e in val_choices:
    val_features_list.append(build_val_feature_matrix(e))
    val_chosen_indices.append(e['chosen_idx'])
    val_customer_ids.append(e['customer_id'])
    val_categories.append(e['category'])

print(f'Val features built: {len(val_features_list):,} events, '
      f'shape per event = {val_features_list[0].shape}')

train_data = {
    'features_list':  train_features_list,
    'chosen_indices': train_chosen_indices,
    'customer_ids':   train_customer_ids,
    'categories':     train_categories,
    'event_weights':  train_event_weights,
}
val_data = {
    'features_list':  val_features_list,
    'chosen_indices': val_chosen_indices,
    'customer_ids':   val_customer_ids,
    'categories':     val_categories,
}

## 3. Smoke test — 2×1×2 mini-grid

Verifies plumbing end-to-end and estimates per-fit wall-clock. If the
estimated full-grid (36 fits) time is > 30 minutes, STOP and do not run the
full grid — report the estimate and let the user decide.

In [ ]:
t0 = time.time()
smoke_best, smoke_results = tune_hyperparameters(
    seed_structure,
    train_data,
    val_data,
    grid_sigma=(5.0, 10.0),
    grid_tau=(1.0,),
    grid_nu=(0.5, 1.0),
    verbose=True,
)
smoke_wall = time.time() - t0
mean_fit = np.mean([r['fit_time'] for r in smoke_results])
est_full = mean_fit * 36
print(f'\nSmoke test: {smoke_wall:.1f}s total  |  mean per fit: {mean_fit:.1f}s')
print(f'Estimated full-grid (36 fits) wall-clock: {est_full/60:.1f} min')
if est_full > 30 * 60:
    print('\n[WARN] Full grid estimated > 30 min — do NOT auto-run the next cell.')
    print('       Decide manually whether to run it (background recommended).')

## 4. Full grid (36 fits)

Skip this cell if the smoke estimate is too long.

In [ ]:
t0 = time.time()
best, results = tune_hyperparameters(
    seed_structure,
    train_data,
    val_data,
    grid_sigma=(5.0, 10.0, 20.0),
    grid_tau=(0.5, 1.0, 2.0),
    grid_nu=(0.25, 0.5, 1.0, 2.0),
    verbose=True,
)
total_wall = time.time() - t0
print(f'\nFull grid wall-clock: {total_wall/60:.1f} min')
print(f'Best: {best}')

## 5. Save tuned hyperparameters

In [ ]:
payload = {
    'best': best,
    'seed_structure': [t.display_name for t in seed_structure.terms],
    'grid': results,
}
out_path = DATA_DIR / 'tuned_hyperparams.json'
with open(out_path, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Wrote {out_path}')

## 6. Sensitivity plot

For each hyperparameter, plot val log-likelihood vs. that parameter, averaging
over the other two axes.

In [ ]:
grid_df = pd.DataFrame(results)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, ['sigma', 'tau', 'nu']):
    agg = grid_df.groupby(col)['val_log_lik'].mean().reset_index()
    ax.plot(agg[col], agg['val_log_lik'], marker='o', color='steelblue')
    ax.set_xlabel(col)
    ax.set_ylabel('mean val log-lik')
    ax.set_title(f'Sensitivity to {col}')
    ax.set_xscale('log')
plt.suptitle(f'Hyperparameter sensitivity — seed = {seed_structure}', y=1.02)
plt.tight_layout()
plot_path = DATA_DIR / 'tuning_sensitivity.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Wrote {plot_path}')

In [ ]:
print(f"Best triple: sigma={best['sigma']}, tau={best['tau']}, nu={best['nu']}")
print(f"Best val log-lik: {best['val_log_lik']:.2f}")

# Interpret flatness of each axis: range of mean val_log_lik across its values.
for col in ['sigma', 'tau', 'nu']:
    agg = grid_df.groupby(col)['val_log_lik'].mean()
    spread = agg.max() - agg.min()
    pretty = {'sigma': r'sigma', 'tau': r'tau', 'nu': r'nu'}[col]
    verdict = 'flat (not sensitive)' if spread < 5.0 else 'peaked (sensitive)'
    print(f'  {pretty}: spread = {spread:.2f}  —  {verdict}')